In [24]:
import pandas as pd 
import ast 
api_results_xls = pd.ExcelFile('retrieval_results/api_retrieved_final.xlsx')
api_results_dct = {} 
for sheet in api_results_xls.sheet_names: 
    api_results_dct[sheet] = pd.read_excel('retrieval_results/api_retrieved_final.xlsx', sheet_name = sheet)


review_question_df = pd.read_excel('PCOS_Guideline_Dataset.xlsm', sheet_name = 'rq_evidence_review')

valid_rq = review_question_df[review_question_df['included_num'] >= 5].copy()
valid_rq_srupdate = valid_rq[valid_rq['sr_update'] == 'Y']
valid_rq_srnew = valid_rq[valid_rq['sr_new'] == 'Y']
valid_rq_srupdate_items = valid_rq_srupdate['Item'].tolist()
valid_rq_srnew_items = valid_rq_srnew['Item'].tolist()



In [27]:
valid_rq_srupdate.columns


Index(['GDG', 'Item', 'Topic', 'Question', 'evidence_review_type', 'sr_update',
       'sr_new', 'included_num', 'searchstrat_year_start',
       'search_strat_yearend', 'searchstrat_notes', 'searchstrat_ovidmedline',
       'searchstrat_pubmed', 'searchstrat_embase', 'searchstrat_psychinfo',
       'searchstrat_cinahl', 'searchstrat_ebmreviews', 'searchstrat_wos',
       'searchstrat_scopus', 'searchstrat_central',
       'searchstrat_clinicaltrialsgov', 'searchstract_whoictrp',
       'searchstrat_googlescholar', 'search_strat_clean_checked',
       'searchstrat_linebylineresults', 'searchstrat_cinahl_flippedorder'],
      dtype='object')

In [29]:

oa_results_df_undupe = api_results_dct['api_results_oa']
overarching_topic_search_articles = oa_results_df_undupe.drop_duplicates(subset = ['api_id_retrieved'])

#consolidate included articles together that match the item list of valid rqs 
srupdate_consolidated = oa_results_df[oa_results_df['question_id'].isin(valid_rq_srupdate_items)]
srnew_consolidated = oa_results_df[oa_results_df['question_id'].isin(valid_rq_srnew_items)]

#columns to join on : question_id with Item, and put in search_strat_year_strat, search_strat_year_end
srupdate_consolidated = srupdate_consolidated.merge(
    valid_rq_srupdate[['Item', 'searchstrat_year_start', 'search_strat_yearend']], 
    left_on='question_id',
    right_on='Item',
    how='left'
)



In [64]:
srupdate_consolidated_valid_articles = srupdate_consolidated[srupdate_consolidated['publication_year'] <= srupdate_consolidated['searchstrat_year_start']]
srupdate_consolidated_valid_articles
# Group by question_id and get top 3 articles by citation_network_size for each new SR
srnew_consolidated_valid_articles_top3 = (srnew_consolidated
    .groupby('question_id')
    .apply(lambda x: x.sort_values('citation_network_size', ascending=False).head(3))
    .reset_index(drop=True)
)

overarching_topic_search_articles = pd.concat([srupdate_consolidated_valid_articles, srnew_consolidated_valid_articles_top3])



C:\Users\dkraj2\AppData\Local\Temp\ipykernel_7952\1591895134.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sort_values('citation_network_size', ascending=False).head(3))


"[{'id': 'https://openalex.org/T10390', 'display_name': 'Diagnosis and Management of Polycystic Ovary Syndrome', 'score': 1.0, 'subfield': {'id': 'https://openalex.org/subfields/2743', 'display_name': 'Reproductive Medicine'}, 'field': {'id': 'https://openalex.org/fields/27', 'display_name': 'Medicine'}, 'domain': {'id': 'https://openalex.org/domains/4', 'display_name': 'Health Sciences'}}, {'id': 'https://openalex.org/T11146', 'display_name': 'Long-Term Effects of Testosterone on Health', 'score': 0.9987, 'subfield': {'id': 'https://openalex.org/subfields/2712', 'display_name': 'Endocrinology, Diabetes and Metabolism'}, 'field': {'id': 'https://openalex.org/fields/27', 'display_name': 'Medicine'}, 'domain': {'id': 'https://openalex.org/domains/4', 'display_name': 'Health Sciences'}}, {'id': 'https://openalex.org/T10364', 'display_name': 'Fertility Preservation in Cancer Patients', 'score': 0.9943, 'subfield': {'id': 'https://openalex.org/subfields/2739', 'display_name': 'Public Health

In [97]:
import json
# Print out types to debug
# Modify to handle Series case explicitly
overarching_topic_search_articles['topic_obj'] = overarching_topic_search_articles['topics'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else ast.literal_eval(x.iloc[0]) if isinstance(x, pd.Series) else x
)

overarching_topic_search_articles['topic_ids'] = overarching_topic_search_articles['topic_obj'].apply(
    lambda x: [d['id'] for d in x] if isinstance(x, list) else None
)

flattened_data = [
    {'original_index': idx, 'topic_id': item} 
    for idx, sublist in overarching_topic_search_articles['topic_ids'].items()
    if isinstance(sublist, list)
    for item in sublist
]

# Convert to DataFrame
flattened_df = pd.DataFrame(flattened_data)

# Group by topic_id and count unique indices
topic_id_coverage = flattened_df.groupby('topic_id')['original_index'].nunique().reset_index()
topic_id_coverage.columns = ['topic_id', 'unique_index_count']

# Calculate total unique indices from flattened_df instead of index
total_unique_indices = len(flattened_df['original_index'].unique())

# Calculate percentage coverage
topic_id_coverage['percentage_coverage'] = (topic_id_coverage['unique_index_count'] / total_unique_indices) * 100
topic_id_coverage_sorted = topic_id_coverage.sort_values(by='percentage_coverage', ascending=False)

In [91]:
flattened_df['topic_id'].nunique()


76

In [98]:
# Group the DataFrame by topic_id and collect the indices for each topic_id
topic_id_to_indices = flattened_df.groupby('topic_id')['original_index'].apply(set).to_dict()

# Initialize a set of all observation indices
all_observations = set(flattened_df['original_index'].unique())

# Initialize an empty set for covered observations and a list to store the selected topic_ids
covered_observations = set()
selected_topic_ids = []

while covered_observations != all_observations:
    # Check if there are any topic_ids left to consider
    if not topic_id_to_indices:
        print("No more topic_ids left to select from.")
        break
    
    # Find the topic_id that covers the most uncovered observations
    best_topic_id = max(topic_id_to_indices, key=lambda k: len(topic_id_to_indices[k] - covered_observations))
    
    # Add this topic_id to the selected list
    selected_topic_ids.append(best_topic_id)
    
    # Update the covered observations
    covered_observations.update(topic_id_to_indices[best_topic_id])
    
    # Remove the selected topic_id from consideration
    del topic_id_to_indices[best_topic_id]

if covered_observations == all_observations:
    print("Selected topic IDs to cover all observations:", selected_topic_ids)
else:
    print("Not all observations could be covered. Selected topic IDs so far:", selected_topic_ids)


Selected topic IDs to cover all observations: ['https://openalex.org/T10390', 'https://openalex.org/T10290', 'https://openalex.org/T10351', 'https://openalex.org/T11144', 'https://openalex.org/T10196', 'https://openalex.org/T10263', 'https://openalex.org/T10459', 'https://openalex.org/T10499', 'https://openalex.org/T10560', 'https://openalex.org/T11888', 'https://openalex.org/T13430']


In [100]:
print(len(selected_topic_ids))

#conduct topic search for selected topic ids for years <= 2022 on OpenAlex 
topic_id = selected_topic_ids[0]
topic_search_path = f"https://api.openalex.org/works?filter=primary_topic.id:{topic_id},type:article|review,from_publication_date:1990-01-01,to_publication_date:2022-09-30&per-page=200&cursor=*"

end_date = '2022-09-30'
start_date = '1990-01-01'  # Fixed typo in variable name

# Create list of dates one year apart
from datetime import datetime, timedelta
date_list = []
current_date = datetime.strptime(start_date, '%Y-%m-%d')
end = datetime.strptime(end_date, '%Y-%m-%d')

while current_date <= end:
    date_list.append(current_date.strftime('%Y-%m-%d'))
    # Add 1 year
    current_date = datetime(current_date.year + 1, current_date.month, current_date.day)



11


In [110]:
from datetime import datetime
from typing import List, Dict

def create_year_ranges(start_date: str = '1980-01-01', 
                      end_date: str = '2022-09-30') -> List[Dict[str, str]]:
    """
    Create list of year-by-year date ranges for OpenAlex API queries
    
    Args:
        start_date: Start date in YYYY-MM-DD format
        end_date: End date in YYYY-MM-DD format
        
    Returns:
        List of dictionaries containing start and end dates for each year
    """
    ranges = []
    current_date = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    
    while current_date < end:
        # Set year end to December 31st of current year
        year_end = datetime(current_date.year, 12, 31)
        
        # If year_end is beyond our end date, use end date instead
        if year_end > end:
            year_end = end
            
        ranges.append({
            'start': current_date.strftime('%Y-%m-%d'),
            'end': year_end.strftime('%Y-%m-%d')
        })
        
        # Move to January 1st of next year
        current_date = datetime(year_end.year + 1, 1, 1)
    
    return ranges

# Create the ranges
searchstrat_start_date = '1980-01-01'
searchstrat_end_date = '2022-09-30'
date_ranges = create_year_ranges(searchstrat_start_date, searchstrat_end_date)
date_ranges


[{'start': '1980-01-01', 'end': '1980-12-31'},
 {'start': '1981-01-01', 'end': '1981-12-31'},
 {'start': '1982-01-01', 'end': '1982-12-31'},
 {'start': '1983-01-01', 'end': '1983-12-31'},
 {'start': '1984-01-01', 'end': '1984-12-31'},
 {'start': '1985-01-01', 'end': '1985-12-31'},
 {'start': '1986-01-01', 'end': '1986-12-31'},
 {'start': '1987-01-01', 'end': '1987-12-31'},
 {'start': '1988-01-01', 'end': '1988-12-31'},
 {'start': '1989-01-01', 'end': '1989-12-31'},
 {'start': '1990-01-01', 'end': '1990-12-31'},
 {'start': '1991-01-01', 'end': '1991-12-31'},
 {'start': '1992-01-01', 'end': '1992-12-31'},
 {'start': '1993-01-01', 'end': '1993-12-31'},
 {'start': '1994-01-01', 'end': '1994-12-31'},
 {'start': '1995-01-01', 'end': '1995-12-31'},
 {'start': '1996-01-01', 'end': '1996-12-31'},
 {'start': '1997-01-01', 'end': '1997-12-31'},
 {'start': '1998-01-01', 'end': '1998-12-31'},
 {'start': '1999-01-01', 'end': '1999-12-31'},
 {'start': '2000-01-01', 'end': '2000-12-31'},
 {'start': '2